In [1]:
from modules.dna_tokenizer import DNATokenizer

# Create tokenizer
tokenizer = DNATokenizer()

# Test
#seq = "ATC---GATCG"
#tokens = tokenizer(seq, return_tensors="pt")
#print(tokens)

/opt/modules/i12g/anaconda/envs/indel_glm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

from modules.bert_config import BertConfig

with open("config.json.txt", "r") as f:
    config_dict = json.load(f)

config = BertConfig(**config_dict)

In [3]:
from modules.custom_bert import DNABertForMaskedLM

model = DNABertForMaskedLM(config)


In [4]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15  # fraction of nucleotides to mask
)

In [5]:
from torch.utils.data import Dataset
import pandas as pd
import torch 

class DNADataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length=512):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        item = self.tokenizer(
                    self.sequences[idx],
                    padding="max_length",
                    #truncation=True,
                    max_length=self.max_length,
                    #return_special_tokens_mask=True,
                )
        return {k: v.squeeze(0) for k, v in item.items()}


data_df = pd.read_parquet("data/simulated/simulated_sequences.parquet")
train_sequences = data_df["sequences"].iloc[:800].tolist()
val_sequences   = data_df["sequences"].iloc[800:1000].tolist()

train_dataset = DNADataset(sequences=train_sequences, tokenizer=tokenizer)
eval_dataset = DNADataset(sequences=val_sequences, tokenizer=tokenizer)



In [6]:
sample = train_dataset[0]
print(sample["input_ids"].shape)


torch.Size([512])


In [7]:
model = DNABertForMaskedLM.from_pretrained("./out/checkpoint-150/")
tokenizer = DNATokenizer.from_pretrained("./out/checkpoint-150/")


In [14]:
inputs = tokenizer("ACGT--ACGT", return_tensors="pt")
outputs = model(**inputs, return_dict=True)
logits = outputs.logits

assert logits.shape[-1] == tokenizer.vocab_size
print("all good")
for o in outputs.logits:
    print(o)
    break

all good
tensor([[-0.8273, -0.7603, -0.8320,  ...,  0.5750,  5.4366,  0.4604],
        [ 2.9217,  2.1711,  2.2629,  ..., -3.0282, -2.9408, -2.5231],
        [ 1.6018,  1.2861,  3.8414,  ..., -2.8236, -2.1940, -2.7212],
        ...,
        [-1.5317, -1.3626, -1.4463,  ..., -1.6434, -1.3318, -1.8684],
        [-1.4622, -1.3858, -1.5105,  ..., -1.6303, -1.3893, -1.7839],
        [-1.4854, -1.4276, -1.5316,  ..., -1.5625, -1.3182, -1.6173]],
       grad_fn=<UnbindBackward0>)
